# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides an end-to-end guide for loading, exploring, and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library, following the structure and best practices established by the Croissant ML data packaging specification.

### Dataset Source

Data is described by its Croissant schema, available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata—fields as attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets and fields, referencing each by its `@id`.

This section lists record set `@id`s and for the first record, shows all available fields and their IDs.

In [ ]:
# Get list of all record sets using @id (Croissant: 'cr:RecordSet')
record_set_objs = [rs for rs in dataset.record_sets]
if not record_set_objs:
    print("No record sets found in this dataset. Please refer to Croissant schema for available record sets.")
else:
    print("Available Record Sets (by @id):")
    for rs in record_set_objs:
        print(f"  - {rs['@id']} (name: {rs.get('name', '(unnamed)')})")
    # Pick the first record set for initial exploration
    sample_rs_id = record_set_objs[0]['@id']

    # Retrieve one record and list its fields and @id's
    print(f"\nFields in record set '{sample_rs_id}':")
    first_record = next(dataset.records(record_set=sample_rs_id), None)
    if first_record:
        for k in first_record.keys():
            print(f"  - Field: {k}")
        sample_fields = list(first_record.keys())
    else:
        print("  (No records found for this record set.)")

## 3. Data Extraction

Here, data from **all record sets** are loaded into DataFrames, each addressed by its record set `@id`.

You can refer to the record set and field `@id`s above for selections or further transformations.

In [ ]:
# Collect all record set @id's (if not already obtained)
if not record_set_objs:
    record_set_objs = [rs for rs in dataset.record_sets]

record_sets_ids = [rs['@id'] for rs in record_set_objs]

dataframes = {}
for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded record set '{rs_id}' into DataFrame with shape {dataframes[rs_id].shape}")
    else:
        print(f"Record set '{rs_id}' contains no records.")

# Show columns from an example record set (using the first with data)
data_rs_ids = [k for k, v in dataframes.items() if not v.empty]
if data_rs_ids:
    example_rs_id = data_rs_ids[0]
    print(f"\nColumns in DataFrame for record set '@id': {example_rs_id}")
    print(dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())
else:
    print("No non-empty record sets found for extraction.")

## 4. Exploratory Data Analysis (EDA)

Let's apply some initial exploratory data analysis to one of the main record sets. Operations include filtering based on a numeric field, normalization, and (if available) grouping by a categorical field.

**Note:** All field/column names are always referenced by their `@id`.


In [ ]:
# Choose a non-empty record set to analyze
if data_rs_ids:
    rs_id = example_rs_id  # Use the first non-empty record set
    df = dataframes[rs_id]

    # Try to find a numeric field by scanning dtypes
    numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if len(numeric_fields) == 0:
        # fallback: look for columns with 'log_likelihood', 'coef', 'odds', 'p_value' in name
        numeric_fields = [c for c in df.columns if any(s in c.lower() for s in ['log_likelihood', 'coef', 'odds', 'p_value', 'score', 'value', 'std', 'mean'])]

    if len(numeric_fields) == 0:
        print(f"No numeric fields found in record set '{rs_id}'. Skipping EDA.")
    else:
        # Select the first numeric field
        numeric_field_id = numeric_fields[0]

        print(f"Using numeric field @id: {numeric_field_id}")

        # Try to filter for values above a sample threshold, using 10 if field min/max unknown
        if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
            threshold = 10
            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())
        else:
            # Try to convert the field to numeric
            df[numeric_field_id + "_num"] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            threshold = 10
            filtered_df = df[df[numeric_field_id + "_num"] > threshold].copy()
            print(f"Filtered records with {numeric_field_id} (converted) > {threshold}:")
            display(filtered_df.head())

        # Normalize the numeric field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try to find a grouping field—use the first string/categorical field
        candidate_group_fields = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) and c != numeric_field_id]
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            print(f"\nGrouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            group_field_id = None
            print("No suitable string/categorical field found for grouping.")
else:
    print("No data available for EDA section.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to the group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Proceed only if we did EDA above
if 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouping field exists, plot means per group
    if 'group_field_id' in locals() and group_field_id:
        if not filtered_df.groupby(group_field_id)[numeric_field_id].mean().empty:
            plt.figure(figsize=(10, 4))
            sns.barplot(
                data=filtered_df,
                x=group_field_id,
                y=numeric_field_id,
                ci=None,
                estimator='mean'
            )
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

- This notebook demonstrates how to load and process a Croissant-packaged dataset from schema using the `mlcroissant` Python library.
- All entities—record sets, fields, and columns—were referred to by their Croissant `@id`.
- Exploratory data analysis included filtering, normalization, and grouping, showing the flexibility of Croissant data packages for research reproducibility.

**For more advanced use (modeling, merging datasets, FAIR data curation), see the [mlcroissant documentation](https://mlcroissant.readthedocs.io/en/latest/).**